In [ ]:
class MorpionEnv:
    def __init__(self):
        self.WIN_MASKS = [7, 56, 448, 73, 146, 292, 273, 84]
        self.reset()
    def reset(self):
        self.board_X = 0
        self.board_O = 0
        return self.board_X, self.board_O
    def get_legal_moves(self):
        occupation = self.board_X | self.board_O
        vides_bits = (~occupation) & 511
        legal_moves = []
        for i in range(9):
            if (vides_bits >> i) & 1:
                legal_moves.append(i)
        return legal_moves
    def step(self, action, player):
        move_mask = 1 << action
        if player == 'X':
            self.board_X |= move_mask
        else:
            self.board_O |= move_mask
    def check_win(self, player_board):
        for mask in self.WIN_MASKS:
            if (player_board & mask) == mask:
                return True
        return False
    def is_draw(self):
        return (self.board_X | self.board_O) == 511

In [ ]:
import random
import json

class QAgent:
    # Paramètres ajustés pour un apprentissage parfait et impitoyable
    def __init__(self, alpha=0.2, gamma=0.95, epsilon=1.0, epsilon_decay=0.99998, min_epsilon=0.01):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

    def get_state(self, board_active, board_waiting):
        return (board_active << 9) | board_waiting

    def get_q_values(self, state):
        if state not in self.q_table:
            self.q_table[state] = [0.0] * 9
        return self.q_table[state]

    def choose_action(self, state, legal_moves):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(legal_moves)
        
        q_values = self.get_q_values(state)
        best_action = legal_moves[0]
        max_q = q_values[best_action]
        
        for action in legal_moves:
            if q_values[action] > max_q:
                max_q = q_values[action]
                best_action = action
        return best_action

    def update_q_value(self, state, action, reward, next_state, next_legal_moves):
        current_q = self.get_q_values(state)[action]
        if next_state is None or len(next_legal_moves) == 0:
            max_future_q = 0.0
        else:
            future_q_values = self.get_q_values(next_state)
            max_future_q = max([future_q_values[m] for m in next_legal_moves])
            
        new_q = current_q + self.alpha * (reward + self.gamma * max_future_q - current_q)
        self.q_table[state][action] = round(new_q, 4)

    def decay_epsilon(self):
        if self.epsilon > self.min_epsilon:
            self.epsilon *= self.epsilon_decay

    def save(self, filename="qtable_morpion.json"):
        with open(filename, 'w') as f:
            json.dump(self.q_table, f)

In [ ]:
def train_self_play(episodes=200000): # Double le temps d'entraînement
    env = MorpionEnv()
    agent = QAgent()
    
    for episode in range(episodes):
        env.reset()
        current_player = 'X'
        prev_state, prev_action = None, None
        
        while True:
            legal_moves = env.get_legal_moves()
            board_active, board_waiting = (env.board_X, env.board_O) if current_player == 'X' else (env.board_O, env.board_X)
            state = agent.get_state(board_active, board_waiting)
            
            action = agent.choose_action(state, legal_moves)
            env.step(action, current_player)
            new_board_active = env.board_X if current_player == 'X' else env.board_O
            
            # Victoire
            if env.check_win(new_board_active):
                agent.update_q_value(state, action, 10, None, [])
                if prev_state is not None:
                    agent.update_q_value(prev_state, prev_action, -10, None, [])
                break
            # Nul
            elif env.is_draw():
                agent.update_q_value(state, action, 0.5, None, [])
                if prev_state is not None:
                    agent.update_q_value(prev_state, prev_action, 0.5, None, [])
                break
            # Continue
            else:
                if prev_state is not None:
                    next_state_for_prev = agent.get_state(board_waiting, new_board_active)
                    agent.update_q_value(prev_state, prev_action, 0, next_state_for_prev, env.get_legal_moves())
            
            prev_state, prev_action = state, action
            current_player = 'O' if current_player == 'X' else 'X'
            
        agent.decay_epsilon()
        
        if (episode + 1) % 40000 == 0:
            print(f"Entraînement : {episode + 1} parties. Epsilon actuel (Hasard) : {agent.epsilon:.3f}")
            
    print(f"Entraînement terminé ! Taille du cerveau (états connus) : {len(agent.q_table)}")
    agent.save("qtable_morpion.json")
    return agent

# Lancer la machine !
trained_agent = train_self_play(200000)

Entraînement : 40000 parties. Epsilon actuel (Hasard) : 0.449
Entraînement : 80000 parties. Epsilon actuel (Hasard) : 0.202
Entraînement : 120000 parties. Epsilon actuel (Hasard) : 0.091
Entraînement : 160000 parties. Epsilon actuel (Hasard) : 0.041
Entraînement : 200000 parties. Epsilon actuel (Hasard) : 0.018
Entraînement terminé ! Taille du cerveau (états connus) : 4520


In [ ]:
class GameInterface:
    def __init__(self, model_file="qtable_morpion.json"):
        with open(model_file, 'r') as f:
            self.q_table = json.load(f)
        self.env = MorpionEnv()
        
    def get_agent_move(self, board_active, board_waiting):
        state = str((board_active << 9) | board_waiting)
        if state in self.q_table:
            q_values = self.q_table[state]
            legal_moves = self.env.get_legal_moves()
            return max(legal_moves, key=lambda m: q_values[m])
        else:
            return random.choice(self.env.get_legal_moves())

    def draw_board(self):
        grid = [' '] * 9
        for i in range(9):
            if (self.env.board_X >> i) & 1: grid[i] = 'X'
            if (self.env.board_O >> i) & 1: grid[i] = 'O'
        
        print(f"\n {grid[6]} | {grid[7]} | {grid[8]} ")
        print("---+---+---")
        print(f" {grid[3]} | {grid[4]} | {grid[5]} ")
        print("---+---+---")
        print(f" {grid[0]} | {grid[1]} | {grid[2]} \n")

    def play(self):
        self.env.reset()
        print("=== NOUVELLE PARTIE ===")
        print("Tu prends les Croix (X) et tu commences. L'IA joue les Ronds (O).")
        
        self.draw_board()
        
        while True:
            # TOUR DU JOUEUR HUMAIN (X)
            while True:
                try:
                    move = int(input("Ton coup en X (0-8) : "))
                    if move in self.env.get_legal_moves():
                        break
                    print("Case déjà prise ou invalide !")
                except ValueError:
                    print("Entre un chiffre valide !")
                    
            self.env.step(move, 'X')
            
            if self.env.check_win(self.env.board_X):
                self.draw_board()
                print("Tu as battu l'IA ! Incroyable. 👑")
                break
            if self.env.is_draw():
                self.draw_board()
                print("Match nul. L'IA a bloqué tes attaques !")
                break

            # TOUR DE L'IA (O)
            print("\nL'IA (O) réfléchit...")
            ia_move = self.get_agent_move(self.env.board_O, self.env.board_X)
            self.env.step(ia_move, 'O')
            self.draw_board()
            
            if self.env.check_win(self.env.board_O):
                print("L'IA a gagné ! Elle a profité de ton erreur. 🤖🏆")
                break
            if self.env.is_draw():
                print("Match nul. L'IA a bloqué tes attaques !")
                break

# Lancer la partie avec l'Humain qui commence
interface = GameInterface()
interface.play()

NameError: name 'json' is not defined